In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers==4.37.2

import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel, TFAutoModelForSequenceClassification
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datetime import datetime

print(tf.__version__)

with open('/content/drive/MyDrive/train_pos_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_pos_content = file.readlines()

with open('/content/drive/MyDrive/train_neg_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_neg_content = file.readlines()

# Create DataFrame
train_pos = pd.DataFrame(train_pos_content, columns=['tweet'])
train_pos['label'] = 1
train_neg = pd.DataFrame(train_neg_content, columns=['tweet'])
train_neg['label'] = 0
train = pd.concat([train_pos, train_neg], ignore_index=True)

# Split data
tweets = train['tweet']
labels = train['label']
train_tweets, val_tweets, train_labels, val_labels = train_test_split(tweets, labels, test_size=0.1, random_state=42)

roberta = "cardiffnlp/twitter-roberta-base-sentiment"

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(roberta)

max_len = 40
X_val_encoded = tokenizer(val_tweets.tolist(),
                          padding=True,
                          truncation=True,
                          max_length=max_len,
                          return_tensors='tf')


X_train_encoded = tokenizer(train_tweets.tolist(),
                            padding=True,
                            truncation=True,
                            max_length=max_len,
                            return_tensors='tf')


# Load RoBERTa model without the classification head
bert_model = TFAutoModel.from_pretrained(roberta)

input_ids = tf.keras.Input(shape=(max_len,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_len,), dtype='int32')
bert_output = bert_model([input_ids, attention_mask])
pooled_output = bert_output.pooler_output

output = tf.keras.layers.Dense(1, activation='sigmoid')(pooled_output)
model = tf.keras.models.Model(inputs=[input_ids, attention_mask], outputs=output)


model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = 2e-5), loss='binary_crossentropy', metrics=['accuracy'])


#checkpoint_path = '/content/drive/MyDrive/Twitter_RoBERTa/cp-0003.ckpt'
#model.load_weights(checkpoint_path)


print(model.summary())
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/Twitter_RoBERTa/cp-{epoch:04d}.ckpt"',  # File path format to save the model
    save_freq='epoch',                      # Save the model after every epoch
    save_weights_only=True,                # Save the entire model (set to True to save only weights)
    verbose=1                               # Verbosity mode, 1 = show messages
)

# Train model
history = model.fit(
    [X_train_encoded['input_ids'], X_train_encoded['attention_mask']],
    train_labels,
    validation_data=(
      [X_val_encoded['input_ids'], X_val_encoded['attention_mask']], val_labels),
    batch_size=256,
    epochs=5,
    callbacks=[checkpoint],
    verbose=1
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 56.0 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.42.4
    Uninstalling transformers-4.42.4:
      Successfully uninstalled transformers-4.42.4
2.15.0


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some layers from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment were not used when initializing TFRobertaModel: ['classifier']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 40)]                 0         []                            
                                                                                                  
 input_2 (InputLayer)        [(None, 40)]                 0         []                            
                                                                                                  
 tf_roberta_model (TFRobert  TFBaseModelOutputWithPooli   1246456   ['input_1[0][0]',             
 aModel)                     ngAndCrossAttentions(last_   32         'input_2[0][0]']             
                             hidden_state=(None, 40, 76                                           
                             8),                                                              

In [5]:

with open('/content/drive/MyDrive/test_cleaned.txt', 'r', encoding='utf-8') as file:
    test_content = file.readlines()

X_test_encoded = tokenizer(test_content,
                            padding=True,
                            truncation=True,
                            max_length=max_len,
                            return_tensors='tf')


y_pred = model.predict([X_test_encoded['input_ids'], X_test_encoded['attention_mask']], verbose = 1, batch_size = 256)

if(len(y_pred) != 10000):
   print('Wrong size')
else:
   y_pred[y_pred <= 0.5] = -1
   y_pred[y_pred > 0.5] = 1
   y_pred = y_pred.astype(int)

   df = pd.read_csv('/content/drive/MyDrive/sample_submission.csv')
   df.Prediction = y_pred

   time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
   filename = "submission" + "_" + time + ".csv"
   store_path = '/content/drive/MyDrive/Submissions/' + filename
   df.to_csv(store_path, index = False)
   print("Submission stored")


40/40 [==============================] - 85s 2s/step
Submission stored
